# dcleaner demo

`dcleaner` is a fluent data-cleaning + visualization layer on top of pandas.
Every method returns the object, so you chain a full
**load → clean → filter → aggregate → plot** pipeline in one expression.

Install: `pip install dcleaner`

In [ ]:
import numpy as np
import pandas as pd
from dclean import Data

## 1. Make a messy dataset

We synthesize a small sales table with missing values and a couple of
out-of-range rows so the cleaning steps have something to do.

In [ ]:
rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    "City": rng.choice(["NY", "LA", "SF", "CHI"], n),
    "Age": rng.integers(16, 65, n),
    "Salary": rng.integers(30000, 130000, n).astype(float),
    "Score": rng.normal(50, 12, n),
})
# inject mess: missing salaries, a couple of under-age rows
df.loc[rng.choice(n, 15, replace=False), "Salary"] = np.nan
df.loc[rng.choice(n, 5, replace=False), "Age"] = rng.integers(10, 15, 5)
df.to_csv("sales.csv", index=False)
df.head()

## 2. Clean + filter in one chain

In [ ]:
d = (Data("sales.csv")
        .dropna()
        .filter("Age > 18"))
d.shape()
d.head()

## 3. Derive a column, then aggregate

In [ ]:
summary = (d
    .mutate(salary_k="Salary / 1000")
    .groupby("City")
    .agg("mean", "Salary"))
summary.df

## 4. Plot it

In [ ]:
(summary
    .plot("bar", x="City", y="Salary", title="Mean salary by city")
    .savefig("salary_by_city.png"))

## 5. Correlation heatmap

In [ ]:
(Data("sales.csv")
    .dropna()
    .plot_corr(title="Feature correlations")
    .savefig("corr.png"))

## 6. Drop back to pandas anytime

In [ ]:
raw = d.to_df()
print(type(raw))
raw.describe()